<div align="right" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/ExploreAI_logos/Logo blue_dark.png"  style="width:25px" align="right";/>
</div>

# Subqueries and CTEs
© ExploreAI Academy

In this exercise, we will apply subqueries and CTEs in different parts of a query and for different use cases. Ensure that you have downloaded the database file, Northwind.db.

## Learning objectives

By the end of this train, you should:
- Know how to use CTEs to simplify subqueries.
- Understand when to use subqueries and when to use CTEs by comparing their performance and readability.

First, let's load our sample database:

In [20]:
# Load and activate the SQL extension to allow us to execute SQL in a Jupyter notebook.
%load_ext sql


The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [21]:
# Load the Northwind database stored in your local machine. 
# Make sure the file is saved in the same folder as this notebook.
%sql sqlite:///Northwind.db
    

Here is a view of all of our tables in the database:

<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/Northwind_ERD.png"  style="width:500px";/>
<br>
<br>
    <em>Figure 1: Northwind ERD</em>
</div>

## Exercise

Run the necessary queries that will provide us with the following information. Compare your queries with the solutions at the end of this notebook.

### Exercise 1

Retrieve product details from products that have been ordered by customers from the UK.

In [22]:
## First find the tables in the Northwind Database
%%sql
SELECT name FROM sqlite_master WHERE type='table';

 * sqlite:///Northwind.db
Done.


name
Categories
Customers
Region
Shippers
Suppliers
Employees
EmployeeTerritories
OrderDetails
Orders
Territories


In [23]:
import pandas as pd
import sqlite3

# Connect to the database
conn = sqlite3.connect('Northwind.db')

# Execute the query to list all tables
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql_query(query, conn)

# Display the tables
tables


,name
0,Categories
1,Customers
2,Region
3,Shippers
4,Suppliers
5,Employees
6,EmployeeTerritories
7,OrderDetails
8,Orders
9,Territories


### Exercise 2


Find out the names of customers who have ordered products of more than the average order value.

In [ ]:
# Add your code here

### Exercise 3


Write a CTE to find the most ordered product by each customer.

In [ ]:
# Add your code here

### Exercise 4

Using a CTE, list employees who have more than the average number of reports.

In [ ]:
# Add your code here

## Solutions

### Exercise 1

SQL solution with a subquery:

In [25]:
%%sql

SELECT customers.*
FROM customers
WHERE customers.CustomerID IN (
    SELECT orders.CustomerID
    FROM orders
    GROUP BY orders.CustomerID
    ORDER BY COUNT(*) DESC
    LIMIT 5
);

 * sqlite:///Northwind.db
Done.


CustomerID,CompanyName,ContactName,ContactTitle,Address,City,Region,PostalCode,Country,Phone,Fax
ERNSH,Ernst Handel,Roland Mendel,Sales Manager,Kirchgasse 6,Graz,None,8010,Austria,7675-3425,7675-3426
FOLKO,Folk och f HB,Maria Larsson,Owner,kergatan 24,Brcke,None,S-844 67,Sweden,0695-34 67 21,None
HUNGO,Hungry Owl All-Night Grocers,Patricia McKenna,Sales Associate,8 Johnstown Road,Cork,Co. Cork,None,Ireland,2967 542,2967 3333
QUICK,QUICK-Stop,Horst Kloss,Accounting Manager,Taucherstrae 10,Cunewalde,None,01307,Germany,0372-035188,None
SAVEA,Save-a-lot Markets,Jose Pavarotti,Sales Representative,187 Suffolk Ln.,Boise,ID,83720,USA,(208) 555-8097,None


SQL solution with a CTE:

In [28]:
%%sql

WITH most_orders AS (
    SELECT orders.CustomerID
    FROM orders
    GROUP BY orders.CustomerID
    ORDER BY COUNT(*) DESC
    LIMIT 5
)
SELECT customers.*
FROM customers
JOIN most_orders
ON customers.CustomerID = most_orders.CustomerID;

 * sqlite:///Northwind.db
Done.


CustomerID,CompanyName,ContactName,ContactTitle,Address,City,Region,PostalCode,Country,Phone,Fax
SAVEA,Save-a-lot Markets,Jose Pavarotti,Sales Representative,187 Suffolk Ln.,Boise,ID,83720,USA,(208) 555-8097,None
ERNSH,Ernst Handel,Roland Mendel,Sales Manager,Kirchgasse 6,Graz,None,8010,Austria,7675-3425,7675-3426
QUICK,QUICK-Stop,Horst Kloss,Accounting Manager,Taucherstrae 10,Cunewalde,None,01307,Germany,0372-035188,None
HUNGO,Hungry Owl All-Night Grocers,Patricia McKenna,Sales Associate,8 Johnstown Road,Cork,Co. Cork,None,Ireland,2967 542,2967 3333
FOLKO,Folk och f HB,Maria Larsson,Owner,kergatan 24,Brcke,None,S-844 67,Sweden,0695-34 67 21,None


Both solutions will return the customer with the most orders. The subquery solution nests the logic inside the main query, which can become difficult to read for more complex queries. The CTE solution separates the logic into a different part of the query, which can be more readable, especially for more complex queries.

### Exercise 2

In [6]:
%%sql

WITH avg_order_value AS (
    SELECT AVG(OrderDetails.UnitPrice * OrderDetails.Quantity) AS average_value
    FROM OrderDetails
)
SELECT DISTINCT customers.CompanyName
FROM customers
JOIN orders ON customers.CustomerID = orders.CustomerID
JOIN OrderDetails ON orders.OrderID = OrderDetails.OrderID
WHERE (OrderDetails.UnitPrice * OrderDetails.Quantity) > (SELECT average_value FROM avg_order_value);


 * sqlite:///Northwind.db
(sqlite3.OperationalError) no such table: customers
[SQL: WITH avg_order_value AS (
    SELECT AVG(OrderDetails.UnitPrice * OrderDetails.Quantity) AS average_value
    FROM OrderDetails
)
SELECT DISTINCT customers.CompanyName
FROM customers
JOIN orders ON customers.CustomerID = orders.CustomerID
JOIN OrderDetails ON orders.OrderID = OrderDetails.OrderID
WHERE (OrderDetails.UnitPrice * OrderDetails.Quantity) > (SELECT average_value FROM avg_order_value);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


### Exercise 3

In [7]:
%%sql

WITH most_ordered_products AS (
    SELECT customers.CustomerID, OrderDetails.ProductID, COUNT(*) AS order_count
    FROM customers
    JOIN orders ON customers.CustomerID = orders.CustomerID
    JOIN OrderDetails ON orders.OrderID = OrderDetails.OrderID
    GROUP BY customers.CustomerID, OrderDetails.ProductID
)
SELECT customers.CompanyName, products.ProductName, max_order_count
FROM (
    SELECT CustomerID, MAX(order_count) AS max_order_count
    FROM most_ordered_products
    GROUP BY CustomerID
) AS max_order_count
JOIN most_ordered_products ON max_order_count.CustomerID = most_ordered_products.CustomerID AND max_order_count.max_order_count = most_ordered_products.order_count
JOIN customers ON most_ordered_products.CustomerID = customers.CustomerID
JOIN products ON most_ordered_products.ProductID = products.ProductID;


 * sqlite:///Northwind.db
(sqlite3.OperationalError) no such table: customers
[SQL: WITH most_ordered_products AS (
    SELECT customers.CustomerID, OrderDetails.ProductID, COUNT(*) AS order_count
    FROM customers
    JOIN orders ON customers.CustomerID = orders.CustomerID
    JOIN OrderDetails ON orders.OrderID = OrderDetails.OrderID
    GROUP BY customers.CustomerID, OrderDetails.ProductID
)
SELECT customers.CompanyName, products.ProductName, max_order_count
FROM (
    SELECT CustomerID, MAX(order_count) AS max_order_count
    FROM most_ordered_products
    GROUP BY CustomerID
) AS max_order_count
JOIN most_ordered_products ON max_order_count.CustomerID = most_ordered_products.CustomerID AND max_order_count.max_order_count = most_ordered_products.order_count
JOIN customers ON most_ordered_products.CustomerID = customers.CustomerID
JOIN products ON most_ordered_products.ProductID = products.ProductID;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


### Exercise 4

In [ ]:
%%sql

WITH avg_reports AS (
    SELECT AVG(report_count) AS average_count
    FROM (
        SELECT COUNT(*) AS report_count
        FROM employees
        JOIN employees AS reports ON employees.EmployeeID = reports.ReportsTo
        GROUP BY employees.EmployeeID
    ) AS report_counts
)
SELECT employees.*
FROM employees
JOIN employees AS reports ON employees.EmployeeID = reports.ReportsTo
GROUP BY employees.EmployeeID
HAVING COUNT(*) > (SELECT average_count FROM avg_reports);

<div align="center" style=" font-size: 80%; text-align: center; margin: 0 auto">
<img src="https://raw.githubusercontent.com/Explore-AI/Pictures/master/ExploreAI_logos/EAI_Blue_Dark.png"  style="width:200px";/>
</div>